Functions needed to precondition CG with preconditioner as described in "Fast Iteratively Reweighted Least Squares
Algorithms for Analysis-Based Sparsity
Reconstruction"

In [1]:
import numpy as np
import math

In [10]:
def create_abs(weights, n):
    """
    weights in np array from np.shape(weights) = n_pixels X n_pixels
    """

    a_1 = weights.flatten()
    a_2 = weights.flatten()

    pix_squared = len(a_1)
    for i in range(pix_squared - 1):
        a_1[i] += a_1[i+1]

    for i in range(pix_squared - n):
        a_2[i] += a_2[i + n]
    
    a = a_1 + a_2

    b = (-weights.flatten())[1:]
    c = (-weights.flatten())[n:]
    return a, b, c

print(create_abs(test_array, 3))

(array([ 8, 12, 16, 20, 24, 28, 22, 25, 18]), array([-2, -3, -4, -5, -6, -7, -8, -9]), array([-4, -5, -6, -7, -8, -9]))


In [47]:
def solve_Ux(diag, updiag, offdiag, n, y):
    """
    n is the number of pixels and also a placement of the offdiagonal. If n = 3, the 
    third offdiagonal has elements again.

    Solves Ux = y
    """
    
    N_big = 2*n
    print("N = ",N_big)

    x = np.zeros(N_big)
    x[-1] = y[-1] / diag[-1]    # setting x[N_big - 1]

    for i in range(2, N_big):
        print(N_big - i)
        if N_big - i > n:
            x[N_big - i] = (1 / diag[N_big - i]) * (y[N_big - i] - updiag[N_big - i]*x[N_big - i + 1])
        if N_big - i <= n:
            print(N_big - i, N_big - i + n - 1)
            x[N_big - i] = (1 / diag[N_big - i]) * (y[N_big - i] - 
                                                    updiag[N_big - i]*x[N_big - i + 1] - 
                                                    offdiag[N_big - i - 1] * x[N_big - i + n - 1])
        print("check x", x)
    return x

In [48]:
diag_test = np.array([1, 2, 3, 4, 5, 6])
updiag_test = np.array([1, 2, 3, 4, 5])
offdiag_test = np.array([1, 2, 3])
y_test = np.ones(6)

print(solve_Ux(diag_test, updiag_test, offdiag_test, 3, y_test))

N =  6
4
check x [0.         0.         0.         0.         0.03333333 0.16666667]
3
3 5
check x [0.         0.         0.         0.09166667 0.03333333 0.16666667]
2
2 4
check x [0.         0.         0.21944444 0.09166667 0.03333333 0.16666667]
1
1 3
check x [0.         0.23472222 0.21944444 0.09166667 0.03333333 0.16666667]
[0.         0.23472222 0.21944444 0.09166667 0.03333333 0.16666667]


In [24]:
x = np.zeros(10)
x[9] = 9